# 🛰️ MAE Training for Satellite Imagery on Google Colab
## Complete Step-by-Step Guide

**What you'll do:**
1. ✅ Set up GPU environment
2. ✅ Clone your GitHub repository
3. ✅ Upload training data
4. ✅ Train MAE model with free GPU
5. ✅ Save results to Google Drive

**Expected Time:** 25-35 hours for 100 epochs (can run in background)

---

## ⚙️ STEP 0: Runtime Setup (IMPORTANT!)

**Before running any cells:**
1. Click **Runtime** → **Change runtime type**
2. Select **GPU** (T4 is free)
3. Click **Save**
4. Wait for runtime to restart

✅ You now have FREE access to NVIDIA T4 GPU!

## 📦 STEP 1: Install Dependencies

In [ ]:
# Install PyTorch with CUDA support
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q

# Install other dependencies
!pip install albumentations pillow tensorboard matplotlib numpy tqdm -q

print("✅ All dependencies installed successfully!")

## 🔍 STEP 2: Verify GPU is Working

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\n{'='*60}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Compute Capability: {torch.cuda.get_device_capability()}")
    print(f"CUDA Version: {torch.version.cuda}")
print(f"{'='*60}")

## 💾 STEP 3: Mount Google Drive (For Saving Results)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("\n✅ Google Drive mounted at /content/drive")
print("📁 Your trained models will be saved here automatically!")

## 📥 STEP 4: Clone Your GitHub Repository

In [ ]:
# Clone your repository
!git clone https://github.com/SnehaShetty-18/ml_project.git

# Navigate to ml-engine directory
%cd /content/ml_project/ml-engine

# Create necessary directories
!mkdir -p data/tiles/train
!mkdir -p data/tiles/val
!mkdir -p checkpoints
!mkdir -p runs

print("\n✅ Repository cloned successfully!")
print("📂 Working directory: /content/ml_project/ml-engine")

## 📤 STEP 5: Upload Training Data

**You have TWO options:**

### Option A: Upload from Google Drive (Recommended for large datasets)
If your data is already in Google Drive:
```python
# Copy data from Drive to Colab
!cp -r /content/drive/MyDrive/your_data_folder/tiles/train/* data/tiles/train/
!cp -r /content/drive/MyDrive/your_data_folder/tiles/val/* data/tiles/val/
```

### Option B: Upload Directly (For smaller datasets < 5GB)
Run the cell below:

In [ ]:
# Upload training tiles (run this if uploading manually)
from google.colab import files
import os

print("📤 Upload your training images...")
print("💡 Tip: You can select multiple files at once")
print("💡 For best results, upload .png or .jpg files")

# Create upload directories
os.makedirs('data/tiles/train', exist_ok=True)
os.makedirs('data/tiles/val', exist_ok=True)

# Upload files
uploaded = files.upload()

# Move uploaded files to train directory
for filename in uploaded.keys():
    !mv {filename} data/tiles/train/

print(f"\n✅ Uploaded {len(uploaded)} files to data/tiles/train/")

## 🔎 STEP 6: Verify Data is Ready

In [ ]:
import os

# Count training and validation images
train_count = len(os.listdir('data/tiles/train')) if os.path.exists('data/tiles/train') else 0
val_count = len(os.listdir('data/tiles/val')) if os.path.exists('data/tiles/val') else 0

print(f"{'='*60}")
print(f"📊 Dataset Summary:")
print(f"   Training images: {train_count:,}")
print(f"   Validation images: {val_count:,}")
print(f"   Total images: {train_count + val_count:,}")
print(f"{'='*60}")

if train_count == 0:
    print("\n⚠️  WARNING: No training images found!")
    print("Please upload your data using Step 5 before training.")
else:
    print("\n✅ Data is ready for training!")

## 🧪 STEP 7: Quick Test (Optional but Recommended)

In [ ]:
# Test if the model can load and run forward pass
import sys
sys.path.append('.')

import torch
from models.mae_model import MAE

print("Testing model initialization...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create a small test model
model = MAE(
    img_size=128,
    patch_size=16,
    embed_dim=768,
    depth=12,
    num_heads=12,
    decoder_embed_dim=512,
    decoder_depth=8,
    decoder_num_heads=16,
    mlp_ratio=4.0,
    norm_layer=torch.nn.LayerNorm
).to(device)

# Test forward pass
x = torch.randn(2, 3, 128, 128).to(device)
with torch.no_grad():
    loss, pred, mask = model(x, mask_ratio=0.75)

print(f"✅ Model test passed!")
print(f"   Input shape: {x.shape}")
print(f"   Loss: {loss.item():.4f}")
print(f"   Prediction shape: {pred.shape}")
print(f"   Mask ratio: {mask.mean().item():.2%}")

## 🚀 STEP 8: Start Training!

**Training Configuration:**
- **Batch Size:** 64 (adjust based on GPU memory)
- **Epochs:** 100 (recommended minimum)
- **Image Size:** 128x128 pixels
- **Learning Rate:** 1e-4 (auto-adjusted)
- **Device:** CUDA GPU

**Estimated Time:** ~25-35 hours for 100 epochs

**💡 Pro Tips:**
- Training will auto-save checkpoints every epoch
- You can close the browser - training continues!
- Check progress in the `runs/` folder
- Models save to `checkpoints/mae_encoder.pth`

In [ ]:
# Start MAE training
!python training/train_mae.py \
  --train-dir data/tiles/train \
  --val-dir data/tiles/val \
  --batch-size 64 \
  --epochs 100 \
  --img-size 128 \
  --device cuda \
  --workers 4 \
  --lr 1e-4 \
  --warmup-epochs 10 \
  --save-every 10

print("\n" + "="*70)
print("🎉 TRAINING COMPLETE!")
print("📁 Model saved to: checkpoints/mae_encoder.pth")
print("📊 Logs saved to: runs/")
print("="*70)

## 💾 STEP 9: Save Results to Google Drive

After training completes, save everything to your Google Drive so you don't lose it!

In [ ]:
import shutil
from datetime import datetime

# Create timestamp for organized storage
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
drive_folder = f'/content/drive/MyDrive/MAE_Training_{timestamp}'

# Create folder in Drive
!mkdir -p {drive_folder}

# Copy trained model
if os.path.exists('checkpoints/mae_encoder.pth'):
    !cp checkpoints/mae_encoder.pth {drive_folder}/
    print(f"✅ Model saved to: {drive_folder}/mae_encoder.pth")

# Copy training logs and TensorBoard runs
if os.path.exists('runs'):
    !cp -r runs {drive_folder}/runs
    print(f"✅ Training logs saved to: {drive_folder}/runs/")

# Copy configuration files
!cp requirements.txt {drive_folder}/
!cp utils/config.py {drive_folder}/
print(f"✅ Config files saved to: {drive_folder}/")

print(f"\n{'='*70}")
print(f"📦 All results saved to Google Drive!")
print(f"📂 Location: My Drive/MAE_Training_{timestamp}/")
print(f"{'='*70}")

## 📊 STEP 10: Monitor Training Progress (Optional)

You can monitor training in real-time using TensorBoard:

In [ ]:
# Load TensorBoard extension
%load_ext tensorboard

# Start TensorBoard
%tensorboard --logdir runs

print("✅ TensorBoard started! Check the visualization above.")
print("💡 You'll see loss curves, learning rate, and training metrics.")

## 🎯 What's Next After Training?

Once training is complete, you can:

1. **Extract Features:** Use the trained encoder to extract features from new satellite images
2. **Analyze Regions:** Perform clustering and analysis on specific geographic regions
3. **Fine-tune:** Adapt the model for specific tasks (classification, segmentation)
4. **Deploy:** Use the model for inference on new data

Check out these scripts in your repository:
- `inference/extract_features.py` - Extract feature vectors
- `inference/analyze_region.py` - Regional analysis
- `analyze_condition.py` - Environmental condition analysis

---

## ❓ Troubleshooting

**Problem:** Out of Memory Error
- **Solution:** Reduce batch size to 32 or 16

**Problem:** Training too slow
- **Solution:** Ensure GPU is selected (Runtime → Change runtime type → GPU)

**Problem:** Colab disconnects
- **Solution:** Training auto-saves every epoch. Just reconnect and resume from last checkpoint.

**Problem:** Can't find data
- **Solution:** Verify files are in `data/tiles/train/` and `data/tiles/val/`

---

## 📚 Additional Resources

- **GitHub Repo:** https://github.com/SnehaShetty-18/ml_project
- **Documentation:** Check README.md and guide files in the repository
- **Issues:** Report bugs or ask questions via GitHub Issues

---

**Happy Training! 🚀🛰️**